In [ ]:
# -*- coding: utf-8 -*-
"""
Incendis forestals a Catalunya (1998-2024) — Visualitzacions Finals Q1-Q4
Information Visualization | Projecte 1
"""

import copy
import json
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# %% ===================================================================
# 0. CÀRREGA I NETEJA DE DADES
# =======================================================================
DATA_PATH = "data/incendis_final.csv"
COMARQUES_MAP_PATH = 'data/divisions-administratives-v2r2-comarques-500000-20260120.json'

df = pd.read_csv(DATA_PATH)
top_causes = df['Causa'].value_counts().nlargest(6).index
df['CausaAgrupada'] = df['Causa'].where(df['Causa'].isin(top_causes), 'Altres')
causa_domain = sorted(df['CausaAgrupada'].unique())
causa_scale = alt.Scale(domain=causa_domain, scheme='dark2')


# --- GeoJSON i càlcul d'aspect ratio real ---
geojson_path = (
    COMARQUES_MAP_PATH
)
with open(geojson_path, encoding='utf-8') as f:
    geo_dict = json.load(f)
comarques_geo = alt.Data(values=geo_dict['features'])
noms_geojson = {feat['properties']['NOMCOMAR'] for feat in geo_dict['features']}


def _calcular_bbox(geo_d):
    xs, ys = [], []

    def recorre(coords):
        if isinstance(coords[0], (int, float)):
            xs.append(coords[0])
            ys.append(coords[1])
        else:
            for c in coords:
                recorre(c)

    for feat in geo_d['features']:
        recorre(feat['geometry']['coordinates'])
    return min(xs), max(xs), min(ys), max(ys)


_minx, _maxx, _miny, _maxy = _calcular_bbox(geo_dict)
ASPECT_RATIO = (_maxx - _minx) / (_maxy - _miny)


def mida_mapa(amplada):
    return amplada, round(amplada / ASPECT_RATIO)


PROJECCIO = 'identity'
REFLECTEIX_Y = True


def titol_rt(titol, subtitol):
    return alt.TitleParams(
        text=titol,
        subtitle=subtitol,
        anchor='start',
        fontSize=20,
        fontWeight='bold',
        subtitleFontSize=13,
        subtitleColor='#555555',
        offset=12
    )

In [ ]:
"""Q2 - Evolution of the causes of fires: five Altair proposals.

Input: the cleaned fires table (OpenRefine export) with at least the columns
    Campania, Origen, Causa, SuperficieTotalForestal
Usage in Colab / Streamlit:
    from q2_charts import prepare, chart1, chart2, chart3, chart4, chart5
    d = prepare(df)            # df = your cleaned DataFrame
    chart1(d)                  # etc.
"""
import altair as alt
import pandas as pd

# ---------------------------------------------------------------- palette --
# One colour per cause family, reused in EVERY chart (same colour = same meaning).
# Okabe-Ito colour-blind-safe palette.
FAMILIES = [
    "Natural (lightning)",
    "Negligence & burns",
    "Machinery & power lines",
    "Intentional",
    "Unknown",
    "Reproduced",
]
FAMILY_COLORS = ["#56B4E9", "#E69F00", "#009E73", "#D55E00", "#999999", "#CC79A7"]
FAMILY_SCALE = alt.Scale(domain=FAMILIES, range=FAMILY_COLORS)
FAMILY_BY_PREFIX = {
    "1": FAMILIES[0],
    "2": FAMILIES[1],
    "3": FAMILIES[2],
    "4": FAMILIES[3],
    "5": FAMILIES[4],
    "6": FAMILIES[5],
}
ERA_A = (1998, 2002)   # period compared in chart 4
ERA_B = (2018, 2022)


# ------------------------------------------------------------- data prep --
def prepare(df: pd.DataFrame) -> pd.DataFrame:
    """Return one row per fire with: yr, cause_code, cause, family, ha."""
    d = df.copy()
    # Cause is only available in the Ministry data (1998-2022).
    d = d[d["Causa"].notna()].copy()
    d["yr"] = pd.to_numeric(d["Campania"], errors="coerce")
    d = d.dropna(subset=["yr"])
    d["yr"] = d["yr"].astype(int)

    # trim double spaces, e.g. "[100]  Rayo" -> "[100] Rayo"
    d["cause"] = d["Causa"].str.replace(r"\s+", " ", regex=True).str.strip()
    d["cause_code"] = d["cause"].str.extract(r"\[(\d+)\]")[0]
    # "[290] Causas no intencionales" and "[290] Otras causas no intencionales"
    # are the same code -> one label
    d.loc[d["cause_code"] == "290", "cause"] = "[290] Otras causas no intencionales"

    d["family"] = d["cause_code"].str[0].map(FAMILY_BY_PREFIX)
    d["ha"] = pd.to_numeric(d["SuperficieTotalForestal"], errors="coerce").fillna(0)
    return d[["yr", "cause_code", "cause", "family", "ha"]]


def _by_family(d):
    """Fires and burnt hectares per year x family (missing combinations = 0)."""
    g = d.groupby(["yr", "family"]).agg(n=("ha", "size"), ha=("ha", "sum")).reset_index()
    full = pd.MultiIndex.from_product(
        [sorted(d["yr"].unique()), FAMILIES], names=["yr", "family"]
    )
    return g.set_index(["yr", "family"]).reindex(full, fill_value=0).reset_index()


YEAR_X = alt.X(
    "yr:Q", title="Year",
    scale=alt.Scale(domain=[1997.5, 2022.5], zero=False, nice=False),
    axis=alt.Axis(format="d", tickMinStep=1, labelOverlap=True, values=list(range(1998, 2023, 4))),
)
LEGEND = alt.Legend(title="Cause of fire")


# ------------------------------------------- 1. normalised stacked area ----
def chart1(d):
    g = _by_family(d)
    tot = g.groupby("yr", as_index=False)["n"].sum()

    area = (
        alt.Chart(g)
        .mark_area(opacity=0.95)
        .encode(
            x=YEAR_X.title(None),
            y=alt.Y("n:Q", stack="normalize", title="Share of fires", axis=alt.Axis(format="%")),
            color=alt.Color("family:N", scale=FAMILY_SCALE, legend=LEGEND),
            order=alt.Order("family:N", sort="ascending"),
            tooltip=["yr:Q", "family:N", alt.Tooltip("n:Q", title="Fires")],
        )
        .properties(width=600, height=260, title="Share of fires by cause")
    )
    total = (
        alt.Chart(tot)
        .mark_bar(color="#555")
        .encode(
            x=YEAR_X,
            y=alt.Y("n:Q", title="Total fires"),
            tooltip=["yr:Q", alt.Tooltip("n:Q", title="Total fires")],
        )
        .properties(width=600, height=70)
    )
    return alt.vconcat(area, total, spacing=4).resolve_scale(color="shared")


# ----------------------------------- 2. small multiples + rolling mean -----
def chart2(d):
    g = _by_family(d)
    base = alt.Chart(g).encode(
        x=YEAR_X,
        color=alt.Color("family:N", scale=FAMILY_SCALE, legend=None),
    )
    pts = base.mark_circle(size=25, opacity=0.35).encode(
        y=alt.Y("n:Q", title="Fires per year"),
        tooltip=["yr:Q", "family:N", alt.Tooltip("n:Q", title="Fires")],
    )
    line = (
        base.transform_window(
            roll="mean(n)", frame=[-2, 2], groupby=["family"], sort=[alt.SortField("yr")]
        )
        .mark_line(strokeWidth=3)
        .encode(y="roll:Q", tooltip=["yr:Q", alt.Tooltip("roll:Q", title="5-yr mean", format=".0f")])
    )
    return (
        alt.layer(pts, line)
        .properties(width=210, height=140)
        .facet(
            facet=alt.Facet("family:N", sort=FAMILIES, title=None,
                            header=alt.Header(labelFontSize=12, labelFontWeight="bold")),
            columns=3,
            title="Fires per year by cause (dots = year, line = 5-year mean)",
        )
        .resolve_scale(y="shared")   # same y axis in every panel -> comparable
    )


# ---------------------------------- 3. heatmap detailed cause x year -------
def chart3(d):
    g = d.groupby(["yr", "cause", "family"]).size().rename("n").reset_index()
    # order rows: by family, then by total fires (largest first)
    tot = d.groupby(["family", "cause"]).size().rename("t").reset_index()
    tot["f"] = tot["family"].map({f: i for i, f in enumerate(FAMILIES)})
    order = tot.sort_values(["f", "t"], ascending=[True, False])["cause"].tolist()

    heat = (
        alt.Chart(g)
        .mark_rect(stroke="white", strokeWidth=0.5)
        .encode(
            x=alt.X("yr:O", title="Year", axis=alt.Axis(labelAngle=-90)),
            y=alt.Y("cause:N", sort=order, title=None, axis=alt.Axis(labelLimit=260)),
            color=alt.Color("n:Q", scale=alt.Scale(scheme="viridis", reverse=True),
                            title="Fires"),
            tooltip=["yr:O", "cause:N", "family:N", alt.Tooltip("n:Q", title="Fires")],
        )
        .properties(width=560, height=360, title="Fires per year by detailed cause")
    )
    return heat


# -------------------------------- 4. slope chart: 1998-02 vs 2018-22 -------
def _repel(p, min_gap=0.11):
    """Spread labels in log10 space so neighbours are at least min_gap apart."""
    import numpy as np
    out = pd.Series(index=p.index, dtype=float)
    for _, grp in p.groupby("period"):
        g = grp.sort_values("n")
        y = np.log10(g["n"].to_numpy())
        for i in range(1, len(y)):
            y[i] = max(y[i], y[i - 1] + min_gap)
        shift = (y - np.log10(g["n"].to_numpy())).mean() / 2   # recentre a bit
        out[g.index] = 10 ** (y - shift)
    return out


def chart4(d):
    def era(y):
        if ERA_A[0] <= y <= ERA_A[1]:
            return f"{ERA_A[0]}–{ERA_A[1]}"
        if ERA_B[0] <= y <= ERA_B[1]:
            return f"{ERA_B[0]}–{ERA_B[1]}"
        return None

    x = d.assign(period=d["yr"].map(era)).dropna(subset=["period"])
    p = (x.groupby(["period", "family"]).size() / 5).rename("n").reset_index()  # fires/year
    p["n"] = p["n"].round(1)
    a, b = f"{ERA_A[0]}–{ERA_A[1]}", f"{ERA_B[0]}–{ERA_B[1]}"
    piv = p.pivot(index="family", columns="period", values="n")
    p["change"] = p["family"].map(((piv[b] / piv[a] - 1) * 100).round(0))
    p["label"] = p["family"] + "  " + p["n"].map("{:.0f}".format)
    p["ly"] = _repel(p)   # label y (data units) nudged so labels don't overlap

    base = alt.Chart(p).encode(
        x=alt.X("period:O", sort=[a, b], title=None, scale=alt.Scale(paddingOuter=1, paddingInner=1),
                axis=alt.Axis(labelFontSize=13, orient="top")),
        y=alt.Y("n:Q", scale=alt.Scale(type="log", domain=[3, 600]),
                title="Fires per year (log scale)",
                axis=alt.Axis(values=[5, 10, 20, 50, 100, 200, 500], grid=False)),
        color=alt.Color("family:N", scale=FAMILY_SCALE, legend=None),
        detail="family:N",
    )
    line = base.mark_line(strokeWidth=3).encode(
        tooltip=["family:N", "period:O", alt.Tooltip("n:Q", title="Fires/year"),
                 alt.Tooltip("change:Q", title="Change (%)")]
    )
    pts = base.mark_circle(size=90)
    left = base.transform_filter(alt.datum.period == a).mark_text(
        align="right", dx=-10, fontSize=12).encode(y=alt.Y("ly:Q", scale=alt.Scale(type="log", domain=[3, 600])), text="label:N")
    right = base.transform_filter(alt.datum.period == b).mark_text(
        align="left", dx=10, fontSize=12).encode(y=alt.Y("ly:Q", scale=alt.Scale(type="log", domain=[3, 600])), text="label:N")
    return alt.layer(line, pts, left, right).properties(
        width=720, height=360, title="Which causes gained or lost weight? (average fires/year)")


# ------------------------ 5. frequency vs severity bubble matrix -----------
def chart5(d):
    g = _by_family(d)
    g = g[g["n"] > 0]
    return (
        alt.Chart(g)
        .mark_circle(opacity=0.95, stroke="#777", strokeWidth=0.5)
        .encode(
            x=alt.X("yr:O", title="Year", axis=alt.Axis(labelAngle=-90)),
            y=alt.Y("family:N", sort=FAMILIES, title=None),
            size=alt.Size("n:Q", title="Number of fires", scale=alt.Scale(range=[20, 700]),
                          legend=alt.Legend(values=[10, 100, 200, 400])),
            # Sequential scale on purpose: here colour = burnt hectares, NOT a cause
            color=alt.Color("ha:Q", title="Burnt forest ha (log)",
                            scale=alt.Scale(type="symlog", scheme="inferno", reverse=True)),
            tooltip=["yr:O", "family:N", alt.Tooltip("n:Q", title="Fires"),
                     alt.Tooltip("ha:Q", title="Burnt ha", format=",.0f")],
        )
        .properties(width=620, height=250, title="Frequency (size) vs severity (colour) by cause")
    )


# ------------------- 6. wide beeswarm: one dot = `per_dot` fires -------------
def beeswarm_data(d, per_dot=5, cols=7, spread=0.9, layout="stacked"):
    """One row per dot (1 dot = `per_dot` fires; cumulative rounding per year).
    layout="lanes"   : one horizontal lane per cause family; dots form a swarm
                       centred on the lane -> every cause shares a baseline.
    layout="stacked" : dots of all causes packed bottom-up inside each year."""
    dx = spread / cols
    counts_all = d.groupby(["yr", "family"]).size().unstack(fill_value=0).reindex(columns=FAMILIES, fill_value=0)
    dots = {}
    for yr, counts in counts_all.iterrows():
        cum, prev = counts.cumsum(), 0
        for fam in FAMILIES:
            end = int(round(cum[fam] / per_dot))
            dots[(yr, fam)] = (end - prev, int(counts[fam]), prev)   # n dots, fires, first dot index
            prev = end
    lane_rows = max(-(-nd // cols) for nd, _, _ in dots.values())   # tallest swarm, in rows
    lane_h = lane_rows + 2
    out = []
    for (yr, fam), (nd, fires, first) in dots.items():
        nrows = -(-nd // cols) if nd else 0
        for j in range(nd):
            k = first + j if layout == "stacked" else j
            r, c = divmod(k, cols)
            x = yr + (c - (cols - 1) / 2) * dx + (dx / 4 if r % 2 == 0 else -dx / 4)
            if layout == "lanes":
                y = (len(FAMILIES) - 1 - FAMILIES.index(fam)) * lane_h + (r - (nrows - 1) / 2) * 0.866 + lane_h / 2
            else:
                y = r * 0.866
            out.append({"yr": yr, "family": fam, "x": x, "y": y, "fires": fires})
    df = pd.DataFrame(out)
    df.attrs["lane_h"] = lane_h
    return df


def chart6(d, per_dot=5, cols=7, width=1500, layout="stacked"):
    """Wide beeswarm of fires per year, coloured by cause. 1 dot = `per_dot` fires.
    layout="stacked" (default): all causes stacked in one swarm per year.
    layout="lanes": one lane per cause."""
    b = beeswarm_data(d, per_dot, cols, layout=layout)
    lane_h = b.attrs["lane_h"]
    px_year = width / 25
    dx_px = px_year * 0.9 / cols                       # pixel distance between dots
    ymax = (len(FAMILIES) * lane_h) if layout == "lanes" else b["y"].max() + 1
    height = int(ymax * dx_px) + 30
    size = (dx_px * 0.92) ** 2                         # circle area ~ touching dots
    xenc = alt.X("x:Q", title="Year",
                 scale=alt.Scale(domain=[1997.4, 2022.6], nice=False, zero=False),
                 axis=alt.Axis(format="d", values=list(range(1998, 2023)), grid=False,
                               labelAngle=0, labelFontSize=11))
    title = f"Fires per year by cause (1 dot = {per_dot} fires)"
    if layout == "lanes":
        centres = [(len(FAMILIES) - 1 - i) * lane_h + lane_h / 2 for i in range(len(FAMILIES))]
        names = "[" + ",".join(f"'{f}'" for f in FAMILIES) + "]"
        yenc = alt.Y("y:Q", title=None, scale=alt.Scale(domain=[0, ymax], nice=False),
                     axis=alt.Axis(values=centres, ticks=False, domain=False, grid=False,
                                   labelFontSize=12, labelFontWeight="bold",
                                   labelExpr=f"{names}[{len(FAMILIES) - 1} - round((datum.value - {lane_h / 2}) / {lane_h})]"))
        legend = None                                  # lanes are already labelled
        bands = alt.Chart(pd.DataFrame({"y0": [i * lane_h for i in range(0, len(FAMILIES), 2)],
                                        "y1": [i * lane_h + lane_h for i in range(0, len(FAMILIES), 2)]})
                          ).mark_rect(color="#f3f3f3").encode(
            y=alt.Y("y0:Q", scale=alt.Scale(domain=[0, ymax], nice=False)), y2="y1:Q")
    else:
        yenc = alt.Y("y:Q", title="Fires per year (approx.)", scale=alt.Scale(domain=[-1, ymax], nice=False),
                     axis=alt.Axis(grid=True, gridDash=[2, 3],
                                   labelExpr=f"round(datum.value / 0.866 * {cols * per_dot})",
                                   values=[0.866 * v / (cols * per_dot) * 1 for v in range(0, int(ymax / 0.866 * cols * per_dot) + 1, 100 if per_dot <= 5 else 200)]))
        legend = LEGEND
        bands = None
    dots = (
        alt.Chart(b)
        .mark_circle(size=size, opacity=1, stroke="white", strokeWidth=0.4)
        .encode(x=xenc, y=yenc,
                color=alt.Color("family:N", scale=FAMILY_SCALE, legend=legend),
                tooltip=["yr:O", "family:N", alt.Tooltip("fires:Q", title="Fires of this cause that year")])
    )
    chart = bands + dots if bands is not None else dots
    return chart.properties(width=width, height=height, title=title)


# ----------------------------------------------------------------- demo ----
if __name__ == "__main__":
    import sys
    df = pd.read_pickle(sys.argv[1]) if sys.argv[1].endswith(".pkl") else pd.read_csv(sys.argv[1])
    d = prepare(df)
    print(d.shape, d["yr"].min(), d["yr"].max(), d["family"].isna().sum(), "unmapped")
    out = sys.argv[2]
    for i, f in enumerate([chart1, chart2, chart3, chart4, chart5, chart6], 1):
        f(d).save(f"{out}/chart{i}.png", scale_factor=1.5)
        print("ok", i)